In [1]:
import os
import random
import warnings
import gc

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression

SEED = 1337


def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 1200)

TF_AVAILABLE = False
TF_IMPORT_ERROR = "Disabled to avoid protobuf/TensorFlow import crash in this runtime."

print("TensorFlow disabled. Using fallback baseline model.")
print("Reason:", TF_IMPORT_ERROR)



TensorFlow disabled. Using fallback baseline model.
Reason: Disabled to avoid protobuf/TensorFlow import crash in this runtime.


In [2]:
df_train = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/train.csv")
df_test = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/test.csv")
df_submission = pd.read_csv(
    "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)

print(
    f'Training Set Shape = {df_train.shape} - Patients = {df_train["Patient"].nunique()}'
)
print(f"Test Set Shape = {df_test.shape} - Patients = {df_test['Patient'].nunique()}")
print(f"Sample Submission Shape = {df_submission.shape}")
print(df_submission.head())




Training Set Shape = (1394, 7) - Patients = 158
Test Set Shape = (18, 7) - Patients = 18
Sample Submission Shape = (1908, 3)
                   Patient_Week   FVC  Confidence
0  ID00126637202218610655908_-3  2000         100
1  ID00126637202218610655908_-2  2000         100
2  ID00126637202218610655908_-1  2000         100
3   ID00126637202218610655908_0  2000         100
4   ID00126637202218610655908_1  2000         100


In [3]:
class Preprocessor:
    def __init__(
        self, df_train, df_test, df_submission, n_folds, shuffle, resize_shape
    ):
        self.df_train = df_train.copy(deep=True)
        self.df_train.sort_values(by=["Patient", "Weeks"], inplace=True)
        self.df_test = df_test.copy(deep=True)
        self.df_submission = df_submission.copy(deep=True)

        self.n_folds = n_folds
        self.shuffle = shuffle
        self.resize_shape = resize_shape

    def _drop_duplicates(self):
        """Calculate mean FVC/Percent per (Patient, Weeks) and drop duplicates."""
        self.df_train["FVC"] = self.df_train.groupby(["Patient", "Weeks"])[
            "FVC"
        ].transform("mean")
        self.df_train["Percent"] = self.df_train.groupby(["Patient", "Weeks"])[
            "Percent"
        ].transform("mean")
        self.df_train.drop_duplicates(inplace=True)
        self.df_train.reset_index(drop=True, inplace=True)

    def _label_encode(self):
        """Label encode categorical features."""
        for df in [self.df_train, self.df_test]:
            df["Sex"] = df["Sex"].map({"Male": 0, "Female": 1})
            df["SmokingStatus"] = df["SmokingStatus"].map(
                {"Never smoked": 0, "Ex-smoker": 1, "Currently smokes": 2}
            )
            df["Sex"] = df["Sex"].fillna(0).astype(int)
            df["SmokingStatus"] = df["SmokingStatus"].fillna(0).astype(int)

    def _create_folds(self):
        """Create 3 different fold schemes (CV1/CV2/CV3) grouped by Patient."""
        self.df_train["Sex_SmokingStatus"] = (
            self.df_train["Sex"].astype(str)
            + "_"
            + self.df_train["SmokingStatus"].astype(str)
        )

        self.df_train["CV1_Fold"] = 0
        for group in self.df_train["Sex_SmokingStatus"].unique():
            patients = self.df_train[self.df_train["Sex_SmokingStatus"] == group][
                "Patient"
            ].unique()
            patients = patients.copy()
            if self.shuffle:
                np.random.seed(SEED)
                np.random.shuffle(patients)

            for fold, patient_group in enumerate(
                np.array_split(patients, self.n_folds), 1
            ):
                self.df_train.loc[
                    self.df_train["Patient"].isin(patient_group), "CV1_Fold"
                ] = fold

        self.df_train["Intercept"] = 0.0
        self.df_train["Coef"] = 0.0
        for patient_name in self.df_train["Patient"].unique():
            patient_df = self.df_train[self.df_train["Patient"] == patient_name]
            last2 = patient_df["FVC"].values[-2:]

            std = float(last2.std())
            if std == 0.0:
                std = 1.0
            z = (last2 - last2.mean()) / std

            reg = LinearRegression().fit(
                patient_df["Weeks"].values[-2:].reshape(-1, 1),
                z,
            )
            self.df_train.loc[self.df_train["Patient"] == patient_name, "Intercept"] = (
                reg.intercept_
            )
            self.df_train.loc[self.df_train["Patient"] == patient_name, "Coef"] = (
                reg.coef_[0]
            )

        self.df_train["Cluster"] = 2
        self.df_train.loc[self.df_train["Coef"] > 0.4, "Cluster"] = 1
        self.df_train.loc[self.df_train["Coef"] < -0.4, "Cluster"] = 3

        self.df_train["CV2_Fold"] = 0
        for group in sorted(self.df_train["Cluster"].unique()):
            patients = self.df_train[self.df_train["Cluster"] == group][
                "Patient"
            ].unique()
            patients = patients.copy()
            if self.shuffle:
                np.random.seed(SEED)
                np.random.shuffle(patients)

            for fold, patient_group in enumerate(
                np.array_split(patients, self.n_folds), 1
            ):
                self.df_train.loc[
                    self.df_train["Patient"].isin(patient_group), "CV2_Fold"
                ] = fold

        self.df_train["CV3_Fold"] = 0
        patients = self.df_train["Patient"].unique().copy()
        np.random.seed(SEED)
        np.random.shuffle(patients)
        for fold, patient_group in enumerate(np.array_split(patients, self.n_folds), 1):
            self.df_train.loc[
                self.df_train["Patient"].isin(patient_group), "CV3_Fold"
            ] = fold

        self.df_train.drop(
            columns=["Sex_SmokingStatus", "Intercept", "Coef", "Cluster"], inplace=True
        )

        for i in range(1, 4):
            self.df_train[f"CV{i}_Fold"] = self.df_train[f"CV{i}_Fold"].astype(np.uint8)

    def _create_baseline_features(self):
        sub = self.df_submission.copy(deep=True)
        sub["Type"] = "Test"
        sub["Patient"] = (
            sub["Patient_Week"].str.split("_", n=1, expand=True)[0].astype(str)
        )
        sub["Weeks"] = (
            sub["Patient_Week"].str.split("_", n=1, expand=True)[1].astype(int)
        )
        sub = sub.drop(columns=["Patient_Week", "FVC", "Confidence"])

        tr = self.df_train.copy(deep=True)
        tr["Type"] = "Train"
        tr["Weeks_Passed"] = tr["Weeks"] - tr.groupby("Patient")["Weeks"].transform(
            "min"
        )
        tr["FVC_Baseline"] = tr.groupby("Patient")["FVC"].transform("first")

        patient_clin = (
            self.df_test[["Patient", "FVC", "Percent", "Age", "Sex", "SmokingStatus"]]
            .rename(columns={"FVC": "FVC_Baseline"})
            .drop_duplicates("Patient")
        )
        sub = sub.merge(patient_clin, on="Patient", how="left")

        sub["Weeks_Passed"] = sub["Weeks"].astype(float)

        df_all = pd.concat([tr, sub], ignore_index=True, axis=0)

        df_all["Age"] = df_all["Age"] + np.floor(df_all["Weeks_Passed"] / 52.0).astype(
            np.int16
        )

        df_all["Weeks"] = df_all["Weeks"].astype(np.int16)
        df_all["Age"] = df_all["Age"].astype(np.float32)
        df_all["FVC_Baseline"] = df_all["FVC_Baseline"].astype(np.float32)
        df_all["Percent"] = df_all["Percent"].astype(np.float32)
        df_all["Weeks_Passed"] = df_all["Weeks_Passed"].astype(np.float32)
        df_all["Sex"] = df_all["Sex"].astype(np.uint8)
        df_all["SmokingStatus"] = df_all["SmokingStatus"].astype(np.uint8)
        if "FVC" in df_all.columns:
            df_all["FVC"] = df_all["FVC"].astype(np.float32)

        self.df_train = df_all.loc[df_all["Type"] == "Train", :].drop(columns=["Type"])
        self.df_test = df_all.loc[df_all["Type"] == "Test", :].drop(
            columns=["Type", "FVC"]
        )

    def get_data(self):
        self._drop_duplicates()
        self._label_encode()
        self._create_folds()
        self._create_baseline_features()

        print(f"Preprocessed Training Set Shape = {self.df_train.shape}")
        print(f"Preprocessed Test Set Shape = {self.df_test.shape}")
        return self.df_train.copy(deep=True), self.df_test.copy(deep=True)




In [4]:
preprocessor_parameters = {
    "df_train": df_train,
    "df_test": df_test,
    "df_submission": df_submission,
    "n_folds": 2,
    "shuffle": True,
    "resize_shape": (512, 512),
}

preprocessor = Preprocessor(**preprocessor_parameters)
df_train, df_test = preprocessor.get_data()

print(df_train.head())
print(df_test.head())



Preprocessed Training Set Shape = (1387, 12)
Preprocessed Test Set Shape = (1908, 11)
                     Patient  Weeks     FVC    Percent   Age  Sex  SmokingStatus  CV1_Fold  CV2_Fold  CV3_Fold  Weeks_Passed  FVC_Baseline
0  ID00007637202177411956430     -4  2315.0  58.253647  79.0    0              1       1.0       2.0       2.0           0.0        2315.0
1  ID00007637202177411956430      5  2214.0  55.712128  79.0    0              1       1.0       2.0       2.0           9.0        2315.0
2  ID00007637202177411956430      7  2061.0  51.862103  79.0    0              1       1.0       2.0       2.0          11.0        2315.0
3  ID00007637202177411956430      9  2144.0  53.950680  79.0    0              1       1.0       2.0       2.0          13.0        2315.0
4  ID00007637202177411956430     11  2069.0  52.063412  79.0    0              1       1.0       2.0       2.0          15.0        2315.0
                        Patient  Weeks  Percent   Age  Sex  SmokingStatus  CV1_F

In [5]:
seed_everything(SEED)

X_train = df_train.copy(deep=True)
y_train = df_train["FVC"].copy(deep=True)

model_parameters = {
    "model": "Stack",
    "predictors": [
        "Age",
        "Sex",
        "SmokingStatus",
        "FVC_Baseline",
        "Percent",
        "Weeks_Passed",
    ],
    "mlp_parameters": {"lr": 0.0005, "epochs": 350, "batch_size": 2**5},
    "qr_parameters": {
        "quantiles": [0.25, 0.5, 0.75],
        "lr": 0.0005,
        "epochs": 150,
        "batch_size": 2**5,
    },
}

# CHANGE (score-up): test patients are unseen, so per-patient models from training don't transfer.
# Keep the same simple linear-regression core idea, but learn a global slope model on train and
# anchor intercept at each test patient's baseline FVC. This usually improves FVC accuracy and
# thus Laplace log-likelihood without changing evaluation semantics.
train_sorted = df_train.sort_values(["Patient", "Weeks"]).copy()

# Build per-row baseline-week and delta features on train to learn slope from baseline to each visit.
base_week = train_sorted.groupby("Patient")["Weeks"].transform("min").astype(np.float32)
train_sorted["Week_Delta"] = (
    train_sorted["Weeks"].astype(np.float32) - base_week
).astype(np.float32)

# Use interactions with time to let clinical variables affect the progression rate (still linear regression).
# (No new model family; still sklearn LinearRegression.)
X_slope = pd.DataFrame(
    {
        "Week_Delta": train_sorted["Week_Delta"].values,
        "Age_x_Week": (
            train_sorted["Age"].astype(np.float32) * train_sorted["Week_Delta"]
        ).values,
        "Percent_x_Week": (
            train_sorted["Percent"].astype(np.float32) * train_sorted["Week_Delta"]
        ).values,
        "Sex_x_Week": (
            train_sorted["Sex"].astype(np.float32) * train_sorted["Week_Delta"]
        ).values,
        "Smoke_x_Week": (
            train_sorted["SmokingStatus"].astype(np.float32)
            * train_sorted["Week_Delta"]
        ).values,
    }
)

# Target is change-from-baseline FVC; this forces intercept ~0 and makes baseline anchoring consistent.
y_delta = (
    train_sorted["FVC"].astype(np.float32)
    - train_sorted["FVC_Baseline"].astype(np.float32)
).values

global_slope_model = LinearRegression()
global_slope_model.fit(X_slope.values, y_delta)

# Residual spread for confidence calibration (metric-aligned); clipped later at >=70.
delta_hat = global_slope_model.predict(X_slope.values).astype(np.float32)
resid = (y_delta - delta_hat).astype(np.float32)
global_sigma = float(np.std(resid))
if (not np.isfinite(global_sigma)) or global_sigma <= 0:
    global_sigma = 200.0  # safe fallback
global_sigma = max(70.0, global_sigma)

# Predict for test rows
week_delta_test = df_test["Weeks_Passed"].astype(np.float32).values
X_slope_test = np.stack(
    [
        week_delta_test,
        (df_test["Age"].astype(np.float32).values * week_delta_test),
        (df_test["Percent"].astype(np.float32).values * week_delta_test),
        (df_test["Sex"].astype(np.float32).values * week_delta_test),
        (df_test["SmokingStatus"].astype(np.float32).values * week_delta_test),
    ],
    axis=1,
).astype(np.float32)

delta_pred_test = global_slope_model.predict(X_slope_test).astype(np.float32)
fvc_pred = (df_test["FVC_Baseline"].astype(np.float32).values + delta_pred_test).astype(
    np.float32
)

# Confidence: allow mild heteroscedasticity with time but keep minimal/risk-free and metric-aligned.
conf_pred = (global_sigma + 0.5 * np.abs(week_delta_test)).astype(np.float32)
conf_pred = np.maximum(conf_pred, 70.0).astype(np.float32)

for cv in [1, 2, 3]:
    df_test[f"CV{cv}_MLP_FVC_Predictions"] = fvc_pred
    df_test[f"CV{cv}_MLP_Confidence_Predictions"] = conf_pred

# Keep existing OOF placeholders (diagnostic only)
for cv in [1, 2, 3]:
    df_train[f"CV{cv}_MLP_FVC_Predictions"] = (
        df_train["FVC_Baseline"].astype(np.float32).values
    )
    df_train[f"CV{cv}_MLP_Confidence_Predictions"] = np.full(
        len(df_train), 70.0, dtype=np.float32
    )




In [6]:
class SubmissionPipeline:
    def __init__(self, df_train, df_test):
        self.df_train = df_train
        self.df_test = df_test

    def laplace_log_likelihood_metric(self, y_true, y_pred, sigma):
        sigma_clipped = np.maximum(sigma, 70)
        delta_clipped = np.minimum(np.abs(y_true - y_pred), 1000)
        score = -np.sqrt(2) * delta_clipped / sigma_clipped - np.log(
            np.sqrt(2) * sigma_clipped
        )
        return np.mean(score)

    def blend(self, by, model, cv):
        self.df_test["Patient_Week"] = (
            self.df_test["Patient"].astype(str)
            + "_"
            + self.df_test["Weeks"].astype(str)
        )

        if by == "model":
            if model == "MLP":
                for df in [self.df_train, self.df_test]:
                    df[f"{model}_FVC"] = (
                        (df["CV1_MLP_FVC_Predictions"] * 0.34)
                        + (df["CV2_MLP_FVC_Predictions"] * 0.33)
                        + (df["CV3_MLP_FVC_Predictions"] * 0.33)
                    )
                    df[f"{model}_Confidence"] = (
                        (df["CV1_MLP_Confidence_Predictions"] * 0.34)
                        + (df["CV2_MLP_Confidence_Predictions"] * 0.33)
                        + (df["CV3_MLP_Confidence_Predictions"] * 0.33)
                    )

                self.df_train[f"{model}_Confidence"] = np.abs(
                    self.df_train[f"{model}_Confidence"].values
                )
                self.df_test[f"{model}_Confidence"] = np.abs(
                    self.df_test[f"{model}_Confidence"].values
                )

                if "FVC" in self.df_train.columns:
                    score = self.laplace_log_likelihood_metric(
                        self.df_train["FVC"].values,
                        self.df_train[f"{model}_FVC"].values,
                        self.df_train[f"{model}_Confidence"].values,
                    )
                    print(f"MLP Blend OOF Score (diagnostic only): {score:.6}")

                self.df_test["FVC"] = self.df_test[f"{model}_FVC"]
                self.df_test["Confidence"] = self.df_test[f"{model}_Confidence"]

            elif model == "QR":
                quantiles = [0.25, 0.50, 0.75]
                for df in [self.df_train, self.df_test]:
                    df[f"{model}_{quantiles[0]}_FVC"] = (
                        (df[f"CV1_QR_{quantiles[0]}_Predictions"] * 0.34)
                        + (df[f"CV2_QR_{quantiles[0]}_Predictions"] * 0.33)
                        + (df[f"CV3_QR_{quantiles[0]}_Predictions"] * 0.33)
                    )
                    df[f"{model}_{quantiles[1]}_FVC"] = (
                        (df[f"CV1_QR_{quantiles[1]}_Predictions"] * 0.34)
                        + (df[f"CV2_QR_{quantiles[1]}_Predictions"] * 0.33)
                        + (df[f"CV3_QR_{quantiles[1]}_Predictions"] * 0.33)
                    )
                    df[f"{model}_{quantiles[2]}_FVC"] = (
                        (df[f"CV1_QR_{quantiles[2]}_Predictions"] * 0.34)
                        + (df[f"CV2_QR_{quantiles[2]}_Predictions"] * 0.33)
                        + (df[f"CV3_QR_{quantiles[2]}_Predictions"] * 0.33)
                    )

                if "FVC" in self.df_train.columns:
                    score = self.laplace_log_likelihood_metric(
                        self.df_train["FVC"].values,
                        self.df_train[f"{model}_{quantiles[1]}_FVC"].values,
                        (
                            self.df_train[f"{model}_{quantiles[2]}_FVC"].values
                            - self.df_train[f"{model}_{quantiles[0]}_FVC"].values
                        ),
                    )
                    print(f"QR Blend OOF Score: {score:.6}")

                self.df_test["FVC"] = self.df_test[f"{model}_{quantiles[1]}_FVC"]
                self.df_test["Confidence"] = (
                    self.df_test[f"{model}_{quantiles[2]}_FVC"]
                    - self.df_test[f"{model}_{quantiles[0]}_FVC"]
                )

        return self.df_test[["Patient_Week", "FVC", "Confidence"]].copy(deep=True)




In [7]:
sub = SubmissionPipeline(df_train, df_test)

df_submission_out = sub.blend(by="model", model="MLP", cv=None)

df_submission_out["FVC"] = df_submission_out["FVC"].astype(float)
df_submission_out["Confidence"] = np.abs(
    df_submission_out["Confidence"].astype(float)
).clip(lower=70.0)

sample = pd.read_csv(
    "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)[["Patient_Week"]]
df_submission_out = sample.merge(df_submission_out, on="Patient_Week", how="left")

if (
    df_submission_out["FVC"].isna().any()
    or df_submission_out["Confidence"].isna().any()
):
    base_map = df_test.drop_duplicates("Patient")[["Patient", "FVC_Baseline"]].copy()
    tmp = df_submission_out["Patient_Week"].str.split("_", n=1, expand=True)
    df_submission_out["Patient"] = tmp[0]
    df_submission_out = df_submission_out.merge(base_map, on="Patient", how="left")
    df_submission_out["FVC"] = df_submission_out["FVC"].fillna(
        df_submission_out["FVC_Baseline"]
    )
    df_submission_out["Confidence"] = df_submission_out["Confidence"].fillna(70.0)
    df_submission_out.drop(columns=["Patient", "FVC_Baseline"], inplace=True)

df_submission_out = df_submission_out[["Patient_Week", "FVC", "Confidence"]]
df_submission_out.to_csv("submission.csv", index=False)

print("Wrote submission.csv")
print(df_submission_out.head())
print(df_submission_out.shape)
print("Any NA:", df_submission_out.isna().any().to_dict())

MLP Blend OOF Score (diagnostic only): -7.7894
Wrote submission.csv
                   Patient_Week          FVC  Confidence
0  ID00126637202218610655908_-3  2360.066650  210.020203
1  ID00126637202218610655908_-2  2354.997559  209.520203
2  ID00126637202218610655908_-1  2349.928955  209.020218
3   ID00126637202218610655908_0  2344.860352  208.520203
4   ID00126637202218610655908_1  2339.815918  209.020218
(1908, 3)
Any NA: {'Patient_Week': False, 'FVC': False, 'Confidence': False}
